# Nemotron-3-Nano LoRA SFT with Stagefreeze Curriculum

> This notebook starts from `nemotron-sft-lora-with-cot-v2-original.ipynb` and applies the **stagefreeze curriculum** that outperformed the single-stage MLX baseline.

## Curriculum overview

This Kaggle notebook keeps the original HF/PEFT/TRL training stack, but replaces the single mixed-data SFT with a **3-stage curriculum** driven by precomputed CSVs:

1. **Stage1 broad trunk**: `train_split_with_cot_stagefreeze_stage1_broad_v3f_safe_plus_notformula.csv`
2. **Stage2 corrective**: `train_split_with_cot_stagefreeze_stage2_corrective_v1.csv`
3. **Stage2.5 reanchor**: `train_split_with_cot_stagefreeze_stage25_reanchor_nonbit_50_each.csv`

## Why stage-specific CSV files?

The training rows are intentionally materialized as **separate CSVs per stage** instead of one large CSV with notebook-side filtering.
That keeps the notebook focused on training logic only and matches the requirement that the curriculum data should be prepared **outside** the code.

## LoRA strategy

- Build one **union LoRA adapter** over the export-safe target modules:
  - `mixer.in_proj`
  - `mixer.out_proj`
  - `mixer.shared_experts.up_proj`
  - `mixer.shared_experts.down_proj`
  - `mixer.q_proj`
  - `mixer.k_proj`
  - `mixer.v_proj`
  - `mixer.o_proj`
- Train only the **broad trunk** in Stage1.
- Freeze the trunk and train only **attention q/k/v/o** in Stage2 and Stage2.5.

This mirrors the stagefreeze curriculum that improved local320 from `215/320` to `221/320` in the MLX experiments.


In [ ]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})

## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# Install trl if needed (for training mode)
if TRAIN_ON_KAGGLE:
    try:
        import trl
        print(f"trl already installed: {trl.__version__}")
    except ImportError:
        # Try offline install first, then online
        offline_path = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"
        if os.path.exists(offline_path):
            subprocess.run(f"pip install --no-index --find-links={offline_path} trl", shell=True)
        else:
            subprocess.run("pip install trl", shell=True)
        import trl
        print(f"trl installed: {trl.__version__}")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, importlib.util, os, subprocess, sys, types

    def sh(cmd: str, check: bool = True):
        print("+", cmd)
        return subprocess.run(cmd, shell=True, check=check)

    def find_spec(name: str) -> bool:
        return importlib.util.find_spec(name) is not None

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    all_datasets = recursive_wheels("datasets-*.whl")
    all_trl = recursive_wheels("trl-*.whl")
    all_multiprocess = recursive_wheels("multiprocess-*.whl")
    all_dill = recursive_wheels("dill-*.whl")
    all_xxhash = recursive_wheels("xxhash-*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    abi_tag = "cxx11abiTRUE" if torch.compiled_with_cxx11_abi() else "cxx11abiFALSE"

    print("Python:", sys.version)
    print("Torch: ", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)
    print("Wheel selector:", {"py_tag": py_tag, "torch": torch_mm, "abi": abi_tag})

    if not torch.cuda.is_available():
        raise RuntimeError(
            "TRAIN_ON_KAGGLE=1 requires GPU runtime because mamba_ssm wheel is CUDA-based."
        )

    def pick_best(wheels):
        exact = [w for w in wheels if py_tag in w and f"torch{torch_mm}" in w and abi_tag in w]
        if exact:
            return exact[-1]
        py_only = [w for w in wheels if py_tag in w]
        if py_only:
            return py_only[-1]
        return None

    if not find_spec("datasets"):
        w = pick_best(all_datasets)
        if w:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
    if not find_spec("trl"):
        w = pick_best(all_trl)
        if w:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
    for pkg, wheels in [("multiprocess", all_multiprocess), ("dill", all_dill), ("xxhash", all_xxhash)]:
        if not find_spec(pkg):
            w = pick_best(wheels)
            if w:
                sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"', check=False)

    if not find_spec("mamba_ssm"):
        causal_wheel = pick_best(all_causal)
        mamba_wheel = pick_best(all_mamba)

        print("Selected causal wheel:", causal_wheel)
        print("Selected mamba wheel:", mamba_wheel)

        if causal_wheel:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{causal_wheel}"')
        if mamba_wheel:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{mamba_wheel}"')
        else:
            raise FileNotFoundError(
                f"No compatible mamba_ssm wheel found under /kaggle/input for "
                f"py={py_tag}, torch={torch_mm}, abi={abi_tag}."
            )

    import datasets
    import trl

    for _mod_name in [
        'mamba_ssm.modules.mamba3',
        'mamba_ssm.ops.cute',
        'mamba_ssm.ops.cute.mamba3',
        'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
    ]:
        sys.modules[_mod_name] = types.ModuleType(_mod_name)
    sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

    import mamba_ssm

    print(f'datasets:  {datasets.__version__}')
    print(f'trl:       {trl.__version__}')
    print(f'mamba_ssm: {mamba_ssm.__version__}')
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import kagglehub

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    print("Model loaded.")
else:
    print("USE_PRETRAINED=1: skipping base model/tokenizer loading.")


In [ ]:
if TRAIN_ON_KAGGLE:
    from peft import LoraConfig, get_peft_model, TaskType

    LORA_RANK = 32
    LORA_ALPHA = 32
    STAGEFREEZE_LORA_TARGET_MODULES = [
        "mixer.in_proj",
        "mixer.out_proj",
        "mixer.shared_experts.up_proj",
        "mixer.shared_experts.down_proj",
        "mixer.q_proj",
        "mixer.k_proj",
        "mixer.v_proj",
        "mixer.o_proj",
    ]
    STAGEFREEZE_STAGE_TARGETS = {
        "stage1_broad": [
            "mixer.in_proj",
            "mixer.out_proj",
            "mixer.shared_experts.up_proj",
            "mixer.shared_experts.down_proj",
        ],
        "stage2_corrective": [
            "mixer.q_proj",
            "mixer.k_proj",
            "mixer.v_proj",
            "mixer.o_proj",
        ],
        "stage25_reanchor": [
            "mixer.q_proj",
            "mixer.k_proj",
            "mixer.v_proj",
            "mixer.o_proj",
        ],
    }

    def set_lora_trainable_modules(peft_model, trainable_suffixes):
        suffix_markers = tuple(f".{suffix}.lora_" for suffix in trainable_suffixes)
        trainable_params = 0
        total_lora_params = 0

        for name, param in peft_model.named_parameters():
            if "lora_" not in name:
                param.requires_grad = False
                continue
            total_lora_params += param.numel()
            is_trainable = any(marker in name for marker in suffix_markers)
            param.requires_grad = is_trainable
            if is_trainable:
                trainable_params += param.numel()

        print({
            "trainable_suffixes": list(trainable_suffixes),
            "trainable_params": trainable_params,
            "total_lora_params": total_lora_params,
        })
        peft_model.print_trainable_parameters()

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=STAGEFREEZE_LORA_TARGET_MODULES,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    set_lora_trainable_modules(model, STAGEFREEZE_STAGE_TARGETS["stage1_broad"])
else:
    print("USE_PRETRAINED=1: skipping PEFT model construction.")


## Mode A: Train stagefreeze curriculum on Kaggle

In [ ]:
if TRAIN_ON_KAGGLE:
    import gc
    import json
    import os
    import re
    import shutil
    import time
    from pathlib import Path

    import pandas as pd
    import torch
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig

    SEED = 123
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    DATASET_ROOT = Path(PRETRAINED_ADAPTER_DATASET_PATH)
    OUTPUT_ROOT = Path("/kaggle/working")
    FINAL_ADAPTER_DIR = OUTPUT_ROOT / "sft_adapter"
    REQUIRED_COLUMNS = ["id", "prompt", "answer", "type", "generated_cot"]

    STAGE_CONFIGS = [
        {
            "stage_name": "stage1_broad",
            "csv_name": "train_split_with_cot_stagefreeze_stage1_broad_v3f_safe_plus_notformula.csv",
            "trainable_suffixes": STAGEFREEZE_STAGE_TARGETS["stage1_broad"],
            "num_train_epochs": 2.0,
            "learning_rate": 1e-4,
            "max_length": 4096,
        },
        {
            "stage_name": "stage2_corrective",
            "csv_name": "train_split_with_cot_stagefreeze_stage2_corrective_v1.csv",
            "trainable_suffixes": STAGEFREEZE_STAGE_TARGETS["stage2_corrective"],
            "num_train_epochs": 2.4,
            "learning_rate": 2e-5,
            "max_length": 768,
        },
        {
            "stage_name": "stage25_reanchor",
            "csv_name": "train_split_with_cot_stagefreeze_stage25_reanchor_nonbit_50_each.csv",
            "trainable_suffixes": STAGEFREEZE_STAGE_TARGETS["stage25_reanchor"],
            "num_train_epochs": 0.45,
            "learning_rate": 1e-5,
            "max_length": 1024,
        },
    ]

    def clear_memory():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def load_stage_dataframe(stage_cfg):
        csv_path = DATASET_ROOT / stage_cfg["csv_name"]
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing curriculum CSV: {csv_path}")
        df = pd.read_csv(csv_path)
        missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
        if missing:
            raise ValueError(f"{csv_path} is missing required columns: {missing}")
        cot_series = df["generated_cot"].fillna("").astype(str).str.strip()
        invalid_mask = cot_series.str.len() < 5
        if invalid_mask.any():
            bad_ids = df.loc[invalid_mask, "id"].astype(str).head(5).tolist()
            raise ValueError(
                f"{csv_path} contains blank/short generated_cot rows; sample ids: {bad_ids}"
            )
        print(f"\n[{stage_cfg['stage_name']}] CSV: {csv_path}")
        print(f"[{stage_cfg['stage_name']}] rows: {len(df)}")
        print(df["type"].value_counts().sort_index())
        return csv_path, df

    def build_stage_dataset(stage_cfg, df):
        records = []
        for row in df.itertuples(index=False):
            prompt = str(row.prompt)
            answer = str(row.answer)
            cot = str(row.generated_cot)
            cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
            if len(cot_cleaned.strip()) < 5:
                raise ValueError(
                    f"{stage_cfg['stage_name']} produced an empty CoT after boxed cleanup for id={row.id}"
                )
            user_content = prompt + PROMPT_SUFFIX
            assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
            records.append({
                "messages": [
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content},
                ]
            })
        dataset = HFDataset.from_list(records)
        print(f"[{stage_cfg['stage_name']}] SFT records: {len(records)}")
        return dataset

    stage_summaries = []
    total_start = time.time()
    last_stage_adapter_dir = None

    for stage_cfg in STAGE_CONFIGS:
        print("\n" + "=" * 88)
        print(f"Starting {stage_cfg['stage_name']}")
        print(json.dumps(stage_cfg, indent=2))

        _, train_df = load_stage_dataframe(stage_cfg)
        dataset = build_stage_dataset(stage_cfg, train_df)
        set_lora_trainable_modules(model, stage_cfg["trainable_suffixes"])

        stage_output_dir = OUTPUT_ROOT / f"{stage_cfg['stage_name']}_output"
        stage_adapter_dir = OUTPUT_ROOT / f"{stage_cfg['stage_name']}_adapter"
        if stage_output_dir.exists():
            shutil.rmtree(stage_output_dir)
        if stage_adapter_dir.exists():
            shutil.rmtree(stage_adapter_dir)

        training_args = SFTConfig(
            output_dir=str(stage_output_dir),
            num_train_epochs=stage_cfg["num_train_epochs"],
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=stage_cfg["learning_rate"],
            lr_scheduler_type="cosine",
            warmup_ratio=0.05,
            max_length=stage_cfg["max_length"],
            logging_steps=10,
            save_strategy="no",
            bf16=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            dataloader_num_workers=2,
            remove_unused_columns=False,
            seed=SEED,
            report_to="none",
            packing=False,
        )

        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        stage_start = time.time()
        trainer.train()
        elapsed = time.time() - stage_start

        model.save_pretrained(stage_adapter_dir)
        tokenizer.save_pretrained(stage_adapter_dir)
        last_stage_adapter_dir = stage_adapter_dir

        stage_summary = {
            "stage_name": stage_cfg["stage_name"],
            "csv_name": stage_cfg["csv_name"],
            "rows": int(len(train_df)),
            "type_counts": {k: int(v) for k, v in train_df["type"].value_counts().sort_index().items()},
            "trainable_suffixes": list(stage_cfg["trainable_suffixes"]),
            "num_train_epochs": float(stage_cfg["num_train_epochs"]),
            "learning_rate": float(stage_cfg["learning_rate"]),
            "max_length": int(stage_cfg["max_length"]),
            "elapsed_minutes": round(elapsed / 60.0, 2),
            "adapter_dir": str(stage_adapter_dir),
        }
        stage_summaries.append(stage_summary)
        print(json.dumps(stage_summary, indent=2))

        del trainer
        del dataset
        clear_memory()

    if last_stage_adapter_dir is None:
        raise RuntimeError("No stage was executed; final adapter is missing.")

    if FINAL_ADAPTER_DIR.exists():
        shutil.rmtree(FINAL_ADAPTER_DIR)
    shutil.copytree(last_stage_adapter_dir, FINAL_ADAPTER_DIR)

    curriculum_summary = {
        "base_notebook": "nemotron-sft-lora-with-cot-v2-original.ipynb",
        "curriculum_name": "stagefreeze_v2_broad_then_attention_then_reanchor",
        "stages": stage_summaries,
        "total_elapsed_minutes": round((time.time() - total_start) / 60.0, 2),
        "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    }
    summary_path = OUTPUT_ROOT / "stagefreeze_curriculum_summary.json"
    with open(summary_path, "w") as f:
        json.dump(curriculum_summary, f, indent=2)

    print("\nCurriculum training complete.")
    print(f"Final adapter copied to {FINAL_ADAPTER_DIR}")
    print(f"Curriculum summary saved to {summary_path}")
else:
    print("USE_PRETRAINED=1: skipping Kaggle training flow.")


## Mode B: Load Pre-trained LoRA

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")
